In [12]:
from pprint import pprint
from openai import OpenAI
from PIL import Image
import heapq
import numpy as np
import yaml
import re
import random
from queue import Queue

from helpers.utils import *
from helpers.env_updated import *
from helpers.large_envs import *
from helpers.agent_maps import *

In [13]:
import ast 
seeds = []
with open('seeds/large/bldg4_seeds.txt', 'r') as f:
    seeds = [ast.literal_eval(line) for line in f if line.strip()]

In [14]:
env = Bldg4()
occ_grid = env.occupancy_grid
sem_grid = env.semantic_grid

seen_occupancy = SeenOccupancyGrid(occ_grid)
seen_semantic = SeenSemanticGrid(sem_grid)

k = 5

In [15]:
def run_frontier_agent(seen_occupancy, seen_semantic, start, goal):
    found_goal = False
    steps = 0
    complete_path = []
    agent_pos = start
    while True:
        steps += 1
        # print(f"STEP {steps}:")
        seen_occupancy.update_with_slice(agent_pos, k)
        seen_semantic.update_with_slice(agent_pos, k)
        # print(seen_occupancy.mark_grid(agent_pos))
        # print(seen_semantic.get_slice(agent_pos, k))
        
        goal_pos = seen_semantic.find_label(goal)
        if goal_pos:
            found_goal = True
            break

        if seen_occupancy.is_fully_explored():
            print("No valid path to goal.")
            break

        # if goal is not in slice, find unexplored cell and move towards it
        # path = frontier_exploration(seen_grid, agent_pos)
        # if path:
        #     agent_pos = path[1]
        # else:
        #     print("No path found")
        #     break       
        frontier_cells = frontier_exploration(seen_semantic, agent_pos)
        random_frontier = None
        if not frontier_cells:
            occ_frontiers = seen_occupancy.find_frontier_cells()
            if not occ_frontiers:
                break  # or return -1
            random_frontier = random.choice(occ_frontiers)
        else:
            random_frontier = random.choice(frontier_cells)
        # print(f"Moving to random frontier cell {random_frontier}")
        # find path to frontier
        path = seen_occupancy.astar(agent_pos, random_frontier)
        agent_pos = path[1] if path else agent_pos

    if found_goal:
        # A* to the goal
        goal_pos = seen_semantic.find_label(goal)
        # print(agent_pos, goal_pos)
        last_path = seen_occupancy.astar(agent_pos, goal_pos)
        try:
            total_steps = len(last_path) + 2 + steps
        except:
            total_steps = steps + 2
            # print(f"Found goal in {total_steps} steps.")
        return total_steps
    else:
        # print("No path to goal found.") 
        return -1


In [16]:
with open('seeds/large/bldg4_frontier.txt', 'w') as f:
    for seed in seeds:
        result = run_frontier_agent(seen_occupancy, seen_semantic, seed['start_pos'], seed['target_room'])
        f.write(str(result))
        f.write('\n')
        print(f"Path {seed['start_pos']} -> {seed['target_pos']}: {result}")
        print(f"Path length: {result}") 

Path (27, 36) -> (23, 62): 216
Path length: 216
Path (22, 17) -> (22, 201): 370
Path length: 370
Path (84, 17) -> (18, 132): 185
Path length: 185
Path (19, 115) -> (18, 195): 85
Path length: 85
Path (88, 19) -> (74, 15): 22
Path length: 22
Path (19, 118) -> (18, 208): 97
Path length: 97
Path (21, 214) -> (18, 168): 3
Path length: 3
Path (11, 74) -> (18, 73): 3
Path length: 3
Path (26, 28) -> (23, 71): 52
Path length: 52
Path (22, 35) -> (23, 71): 41
Path length: 41


In [17]:
for seed in seeds:
        print(seed['start_pos'], seed['target_room'])
        result = run_frontier_agent(seen_occupancy, seen_semantic, seed['start_pos'], seed['target_room'])
        print(f"Path {seed['start_pos']} -> {seed['target_pos']}: {result}")
        print(f"Path length: {result}") 

(27, 36) 138
Path (27, 36) -> (23, 62): 36
Path length: 36
(22, 17) 166
Path (22, 17) -> (22, 201): 190
Path length: 190
(84, 17) 149
Path (84, 17) -> (18, 132): 185
Path length: 185
(19, 115) 163
Path (19, 115) -> (18, 195): 85
Path length: 85
(88, 19) 103
Path (88, 19) -> (74, 15): 22
Path length: 22
(19, 118) 167
Path (19, 118) -> (18, 208): 95
Path length: 95
(21, 214) 159
Path (21, 214) -> (18, 168): 3
Path length: 3
(11, 74) 131
Path (11, 74) -> (18, 73): 3
Path length: 3
(26, 28) 140
Path (26, 28) -> (23, 71): 52
Path length: 52
(22, 35) 140
Path (22, 35) -> (23, 71): 41
Path length: 41
